# AHN window diagnostic — GatedDeltaNet, observe-only (Task #1)

Self-contained. **No repo upload, no GitHub.** The minimal `ahn-mdc` project (the validated dataset / scorer / token-accounting + the observe-only diagnostic from branch `saadat-pipeline-validation`, commits `71349c8` + `4cfd9a3`) is embedded below as a base64 tarball and reconstructed into `/kaggle/working/ahn-mdc`.

The Juan-approved matched inference configuration is baked into the bundled `src/ahnexp/models.py` (`_force_window`): **sliding_window=256, sliding_window_type=fixed, ahn_position=prefix, num_attn_sinks=0**, stale `dy_*` keys deleted. The diagnostic itself is **observe-only** — it never writes `model.config`.

**Kaggle settings:** Accelerator = GPU (T4 x2 or P100) · Internet = On · Persistence = Files only.

## Run order
| step | cells | note |
|---|---|---|
| PART A — reconstruct + install | **1, 2** | ~5–10 min |
| **RESTART KERNEL** | — | Run ▸ Restart & clear cell outputs (transformers must be 4.51 before import) |
| PART B — verify + diagnose | **3, 4, 5** | first run ~15–20 min (6 GB base download + one-time merge); exactly 2 generations |

tarball sha256: `9aea2a5597558b6747db45dc2e0007c8e0e9f2ef4d938785187802b8daabc855`


## PART A · Cell 1 — reconstruct the minimal project


In [ ]:
import base64, hashlib, io, tarfile, pathlib

ROOT = "/kaggle/working/ahn-mdc"
EXPECT_SHA = "9aea2a5597558b6747db45dc2e0007c8e0e9f2ef4d938785187802b8daabc855"

_BLOB = (
    "..."
)

blob = base64.b64decode(_BLOB)
assert hashlib.sha256(blob).hexdigest() == EXPECT_SHA, "tarball checksum mismatch"
pathlib.Path(ROOT).mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as tf:
    tf.extractall(ROOT)

got = sorted(str(p.relative_to(ROOT)) for p in pathlib.Path(ROOT).rglob("*") if p.is_file())
print(f"reconstructed {len(got)} files under {ROOT}:")
for p in got:
    print("  ", p)

for need in ("config/experiment.yaml", "config/facts.yaml",
             "src/ahnexp/models.py", "src/ahnexp/dataset.py", "src/ahnexp/evaluate.py",
             "scripts/diag_ahn_window.py"):
    assert (pathlib.Path(ROOT) / need).is_file(), f"missing {need}"

# confirm the bundled loader carries the Juan-approved freeze
mtxt = (pathlib.Path(ROOT) / "src/ahnexp/models.py").read_text()
assert 'model.config.sliding_window_type = matched["sliding_window_type"]' in mtxt
assert "model.config.num_attn_sinks = 0" in mtxt
assert '("dy_sliding_window", "dy_num_attn_sinks")' in mtxt
print("\nOK - bundled models.py carries the matched-config freeze (256 / fixed / prefix / 0).")


## PART A · Cell 2 — environment (pin transformers 4.51, install AHN + flash-attn)


In [ ]:
import subprocess
# runs the bundled, validated setup script:
#   pins transformers==4.51.0, installs the flash-linear-attention fork, wandb,
#   einops, flash-attn==2.8.3, clones ByteDance-Seed/AHN to /kaggle/working/AHN
#   and pip-installs it (core only), plus pyyaml / jinja2 / pyarrow.
rc = subprocess.call(["bash", "/kaggle/working/ahn-mdc/scripts/setup_kaggle.sh"])
print("\nsetup exit code:", rc)
print(">>> NOW RESTART THE KERNEL:  Run menu -> Restart & clear cell outputs  <<<")
print(">>> then run PART B (Cell 3 onward). Do NOT re-run Cell 1 / Cell 2.     <<<")


## ⚠️ RESTART THE KERNEL NOW

**Run ▸ Restart & clear cell outputs.** `transformers` was just pinned to `4.51.0` on disk but the running kernel still holds the Kaggle default; the AHN custom classes only work on 4.51.

After restarting, run **Cell 3, 4, 5** (skip Cell 1 / Cell 2 — the files and packages are already there).


## PART B · Cell 3 — verify the environment (fail fast)


In [ ]:
import importlib, torch, transformers
print("torch        ", torch.__version__, "| cuda build", torch.version.cuda,
      "| cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu          ", torch.cuda.get_device_name(0))
print("transformers ", transformers.__version__)
assert torch.cuda.is_available(), "No GPU - set Accelerator to GPU and restart."
assert transformers.__version__.startswith("4.51"), (
    f"transformers is {transformers.__version__}, need 4.51.x - re-run Cell 2 and RESTART.")
for mod in ("fla", "flash_attn", "ahn.transformer.qwen2_ahn"):
    try:
        importlib.import_module(mod); print(f"import {mod:28} OK")
    except Exception as e:
        print(f"import {mod:28} FAIL: {type(e).__name__}: {e}")
        if mod == "flash_attn":
            print("   -> the AHN self-attention hardcodes FlashAttention-2; inference will fail without it.")
print("\nenvironment looks ready - run Cell 4.")


## PART B · Cell 4 — run the observe-only diagnostic

Loads **only GatedDeltaNet**, merges weights once (~6 GB base download, cached), verifies the custom AHN class + `.ahn` params, builds two trajectories straddling W=256, hard-stops if the recurrent one does not cross W, then runs **exactly one exact-memory and one recurrent-memory generation**. Observe-only: it does not modify `model.config`.


In [ ]:
import subprocess, sys
cmd = [sys.executable, "/kaggle/working/ahn-mdc/scripts/diag_ahn_window.py",
       "--repo", "/kaggle/working/ahn-mdc",
       "--ahn-repo", "/kaggle/working/AHN"]
print(" ".join(cmd), "\n")
rc = subprocess.call(cmd)
print("\ndiagnostic exit code:", rc, "(0 = PASSED, 2 = INCONCLUSIVE, 1 = hard-stop/error)")


## PART B · Cell 5 — print the diagnostic JSON


In [ ]:
import pathlib
p = pathlib.Path("/kaggle/working/ahn-mdc/outputs/diag_ahn_window.json")
if p.is_file():
    print(p.read_text())
else:
    print("no JSON written - the run hard-stopped before the end. Paste the Cell 4 output above.")


## What to paste back for review

The **full output of Cell 4** plus the **Cell 5 JSON**. Key checks:

* `[0]` `import flash_attn OK`; torch ≥ 2.5
* `[2]` checkpoint pre-override: `sliding_window 256`, `sliding_window_type random`, `ahn_position random`
* `[3]` after `models.load`: `sliding_window 256`, **`sliding_window_type fixed`**, **`ahn_position prefix`**, **`num_attn_sinks 0`**, `dy_* <unset>`, `model.training False`, `effective_window 256`
* `[3b]` `model class ahn.transformer.qwen2_ahn.qwen2_ahn.Qwen2ForCausalLM` (NOT `transformers.models.qwen2…`); `generation_config.use_cache True`
* `[2] AHN modules` `36 x Qwen2MemDecoderLayer`, `layer[0] .ahn = BaseAHN / fn=GatedDeltaNet`, `.ahn.` param count
* `[6]` the two `model_tokens_after_target` (≈49 and ≈2100)
* `HARD GATE` line = `PASS`
* `[9]` both trial rows: `prediction`, `answer_canonical`, `correct`, `malformed`, `abstained`, `n_new_tokens`, **`ahn_layer0_num_cached_tokens`** (0 exact / ~1850+ recurrent), **`ahn_kernel_forward_calls`** (0 exact / ≥1 recurrent)
* `[10]` verdict block + `DIAGNOSTIC PASSED` / `INCONCLUSIVE`

If Cell 4 exits early, paste whatever printed — the hard gate stopping is a valid result. **Do not run the experimental grid regardless of outcome.**
